## Предсказание доходов клиентов банка
### Ключевые улучшения относительно исходного решения:

| Аспект | Исходное решение | Улучшенное решение |
|--------|-----------------|-------------------|
| **Модели** | 1 модель (последний fold) | Ансамбль 10 моделей (2 конфигурации × 5 folds) |
| **Feature Engineering** | ~220 признаков | ~400+ признаков |
| **Регуляризация** | Нет L1/L2 | L1=0.1, L2=0.1 |
| **Learning rate** | 0.05 | 0.03 (медленнее, стабильнее) |
| **num_leaves** | 128 | 63 (меньше переобучения) |
| **Early stopping** | 50 rounds | 150 rounds |
| **Blending** | Нет | Поиск оптимальных весов |

**Ожидаемое улучшение WMAE*: ~200-500 единиц**

In [ ]:
# Импорты
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Конфигурация
RANDOM_STATE = 42
N_FOLDS = 5
EARLY_STOPPING_ROUNDS = 150

print("✅ Библиотеки загружены")

In [ ]:
# Загрузка данных (измените пути на свои)
# Для Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

TRAIN_PATH = '/content/drive/MyDrive/Альфа банк_хакатон/hackathon_income_train.csv'
TEST_PATH = '/content/drive/MyDrive/Альфа банк_хакатон/hackathon_income_test.csv'
SAMPLE_SUB_PATH = '/content/drive/MyDrive/Альфа банк_хакатон/sample_submission.csv'

train_df = pd.read_csv(TRAIN_PATH, decimal=',', sep=';', low_memory=False)
test_df = pd.read_csv(TEST_PATH, decimal=',', sep=';', low_memory=False)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Sample submission shape: {sample_sub.shape}")

In [ ]:
# EDA: Распределения target и weights
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Target distribution
axes[0].hist(train_df['target'], bins=100, edgecolor='k', alpha=0.7, color='steelblue')
axes[0].set_title('Distribution of Target (Income)')
axes[0].set_xlabel('Income')

# Log target
axes[1].hist(np.log1p(train_df['target']), bins=100, edgecolor='k', alpha=0.7, color='coral')
axes[1].set_title('Distribution of Log(Target)')
axes[1].set_xlabel('Log(Income)')

# Weights
axes[2].hist(train_df['w'], bins=100, edgecolor='k', alpha=0.7, color='seagreen')
axes[2].set_title('Distribution of Weights')
axes[2].set_xlabel('Weight')

plt.tight_layout()
plt.show()

print(f"Target: mean={train_df['target'].mean():.0f}, median={train_df['target'].median():.0f}, std={train_df['target'].std():.0f}")
print(f"Weights: mean={train_df['w'].mean():.3f}, std={train_df['w'].std():.3f}")

In [ ]:
def advanced_feature_engineering(df, is_train=True):
    """
    Расширенный Feature Engineering
    Создает ~180 новых признаков поверх исходных ~220
    """
    df = df.copy()
    
    # 1. Конвертация object -> numeric
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 2. ЛОГАРИФМИРОВАНИЕ денежных признаков
    money_keywords = ['sum', 'amt', 'turn', 'limit', 'salary', 'income', 'avg', 'max', 'payout']
    money_cols = [c for c in df.columns if any(kw in c.lower() for kw in money_keywords)]
    for col in money_cols:
        if col in df.columns and df[col].dtype in ['float64', 'int64']:
            df[f'log_{col}'] = np.log1p(df[col].clip(lower=0).fillna(0))
    
    # 3. ОТНОШЕНИЯ между признаками
    ratio_pairs = [
        ('turn_cur_cr_sum_v2', 'turn_cur_db_sum_v2'),
        ('turn_cur_cr_avg_v2', 'turn_cur_db_avg_v2'),
        ('turn_cur_cr_max_v2', 'turn_cur_db_max_v2'),
        ('avg_cur_cr_turn', 'avg_cur_db_turn'),
        ('turn_cur_cr_avg_v2', 'turn_cur_cr_max_v2'),
    ]
    for num_col, den_col in ratio_pairs:
        if num_col in df.columns and den_col in df.columns:
            df[f'ratio_{num_col[:15]}_{den_col[:15]}'] = df[num_col] / (df[den_col] + 1e-8)
    
    # 4. РАЗНОСТИ
    if 'turn_cur_cr_sum_v2' in df.columns and 'turn_cur_db_sum_v2' in df.columns:
        df['diff_cr_db_sum'] = df['turn_cur_cr_sum_v2'] - df['turn_cur_db_sum_v2']
        df['diff_cr_db_abs'] = np.abs(df['diff_cr_db_sum'])
    
    # 5. АГРЕГАЦИИ по категориям трат
    category_groups = {
        'grocery': ['supermarkety', 'gipermarkety', 'produkty'],
        'transport': ['transport', 'taksi', 'avto', 'toplivo'],
        'entertainment': ['restorany', 'kino', 'razvlechenija'],
        'finance': ['bankomat', 'perevod', 'platezh', 'vydacha'],
        'health': ['apteki', 'medicina'],
        'shopping': ['odezhda', 'obuv'],
    }
    for group_name, keywords in category_groups.items():
        group_cols = [c for c in df.columns if any(kw in c.lower() for kw in keywords)]
        if group_cols:
            df[f'total_{group_name}'] = df[group_cols].fillna(0).sum(axis=1)
            df[f'mean_{group_name}'] = df[group_cols].fillna(0).mean(axis=1)
            df[f'std_{group_name}'] = df[group_cols].fillna(0).std(axis=1)
            df[f'count_nonzero_{group_name}'] = (df[group_cols].fillna(0) > 0).sum(axis=1)
    
    # 6. БКИ агрегации
    bki_cols = [c for c in df.columns if 'bki' in c.lower()]
    if bki_cols:
        df['bki_mean'] = df[bki_cols].mean(axis=1)
        df['bki_std'] = df[bki_cols].std(axis=1)
        df['bki_max'] = df[bki_cols].max(axis=1)
        df['bki_sum'] = df[bki_cols].sum(axis=1)
        df['bki_count_positive'] = (df[bki_cols] > 0).sum(axis=1)
    
    # 7. Цифровой профиль (dp_) агрегации
    dp_cols = [c for c in df.columns if c.startswith('dp_')]
    if dp_cols:
        df['dp_mean'] = df[dp_cols].mean(axis=1)
        df['dp_std'] = df[dp_cols].std(axis=1)
        df['dp_max'] = df[dp_cols].max(axis=1)
    
    # 8. Индикаторы пропусков
    important_cols = ['salary_6to12m_avg', 'first_salary_income', 'incomeValue',
                      'turn_cur_cr_sum_v2', 'hdb_bki_total_max_limit']
    for col in important_cols:
        if col in df.columns:
            df[f'{col}_isna'] = df[col].isna().astype(int)
    
    # 9. Квантильные бины
    top_features = ['salary_6to12m_avg', 'first_salary_income', 'turn_cur_cr_avg_act_v2']
    for col in top_features:
        if col in df.columns:
            try:
                df[f'{col}_qbin'] = pd.qcut(df[col].rank(method='first'), q=10, labels=False, duplicates='drop')
            except:
                pass
    
    # 10. Взаимодействия
    if 'salary_6to12m_avg' in df.columns and 'turn_cur_cr_avg_act_v2' in df.columns:
        df['salary_x_turn_cr'] = df['salary_6to12m_avg'] * df['turn_cur_cr_avg_act_v2'] / 1e10
    if 'age' in df.columns and 'salary_6to12m_avg' in df.columns:
        df['salary_per_age'] = df['salary_6to12m_avg'] / (df['age'] + 1)
    
    # 11. Категориальные признаки
    cat_cols = ['gender', 'adminarea', 'city_smart_name']
    for col in cat_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
    
    return df

print("✅ Функция feature engineering определена")

In [ ]:
# Применяем Feature Engineering
print("Применение FE к train...")
train_fe = advanced_feature_engineering(train_df)
print("Применение FE к test...")
test_fe = advanced_feature_engineering(test_df, is_train=False)

# Выбираем признаки
exclude_cols = ['id', 'dt', 'target', 'w', 'incomeValue', 'incomeValueCategory']
feature_cols = [c for c in train_fe.columns 
                if c not in exclude_cols 
                and train_fe[c].dtype in ['float64', 'int64', 'float32', 'int32']]

# Выравниваем колонки test
for col in feature_cols:
    if col not in test_fe.columns:
        test_fe[col] = 0

print(f"\n📊 Количество признаков:")
print(f"   Исходных: ~220")
print(f"   После FE: {len(feature_cols)}")

# Подготовка данных
X = train_fe[feature_cols].astype(np.float32).fillna(-999)
y = train_fe['target'].values
w = train_fe['w'].values
X_test = test_fe[feature_cols].astype(np.float32).fillna(-999)

print(f"\n📐 Размеры:")
print(f"   X_train: {X.shape}")
print(f"   X_test: {X_test.shape}")

In [ ]:
# Метрика WMAE*
def wmae_score(y_true, y_pred, weights):
    """Weighted Mean Absolute Error (метрика хакатона)"""
    return np.mean(weights * np.abs(y_true - y_pred))

# Baseline
baseline_pred = y.mean()
baseline_wmae = wmae_score(y, np.full_like(y, baseline_pred), w)
print(f"📏 Baseline (mean prediction) WMAE*: {baseline_wmae:.2f}")

In [ ]:
# ============================================================
# МОДЕЛЬ 1: LightGBM с оптимизированными параметрами
# ============================================================
print("=" * 60)
print("МОДЕЛЬ 1: LightGBM (оптимизированные параметры)")
print("=" * 60)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

models_lgb1 = []
scores_lgb1 = []
oof_lgb1 = np.zeros(len(X))

# Оптимизированные параметры (vs исходные)
params_lgb1 = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'num_leaves': 63,           # было 128 → меньше переобучения
    'learning_rate': 0.03,      # было 0.05 → медленнее, стабильнее
    'feature_fraction': 0.7,    # было 0.9 → больше разнообразия
    'bagging_fraction': 0.7,    # было 0.8
    'bagging_freq': 5,
    'min_child_samples': 30,    # было 20 → больше данных в листе
    'lambda_l1': 0.1,           # НОВОЕ: L1 регуляризация
    'lambda_l2': 0.1,           # НОВОЕ: L2 регуляризация
    'verbose': -1,
    'seed': RANDOM_STATE,
    'n_jobs': -1
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n🔄 Fold {fold + 1}/{N_FOLDS}")
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    w_tr, w_val = w[tr_idx], w[val_idx]
    
    train_data = lgb.Dataset(X_tr, label=y_tr, weight=w_tr)
    valid_data = lgb.Dataset(X_val, label=y_val, weight=w_val, reference=train_data)
    
    model = lgb.train(
        params_lgb1,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=3000,    # больше итераций
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING_ROUNDS),  # было 50 → 150
            lgb.log_evaluation(100)
        ]
    )
    
    pred = model.predict(X_val)
    oof_lgb1[val_idx] = pred
    score = wmae_score(y_val, pred, w_val)
    scores_lgb1.append(score)
    models_lgb1.append(model)
    print(f"   WMAE*: {score:.4f}")

print(f"\n🎯 LightGBM-1 Mean WMAE*: {np.mean(scores_lgb1):.4f} ± {np.std(scores_lgb1):.4f}")
print(f"🎯 LightGBM-1 OOF WMAE*:  {wmae_score(y, oof_lgb1, w):.4f}")

In [ ]:
# ============================================================
# МОДЕЛЬ 2: LightGBM с альтернативными параметрами
# ============================================================
print("=" * 60)
print("МОДЕЛЬ 2: LightGBM (альтернативные параметры для diversity)")
print("=" * 60)

models_lgb2 = []
scores_lgb2 = []
oof_lgb2 = np.zeros(len(X))

# Альтернативные параметры для разнообразия ансамбля
params_lgb2 = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'num_leaves': 127,          # больше листьев
    'learning_rate': 0.02,      # ещё медленнее
    'feature_fraction': 0.6,
    'bagging_fraction': 0.6,
    'bagging_freq': 3,
    'min_child_samples': 50,
    'max_depth': 10,            # ограничение глубины
    'verbose': -1,
    'seed': 123,                # другой seed для разнообразия
    'n_jobs': -1
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n🔄 Fold {fold + 1}/{N_FOLDS}")
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    w_tr, w_val = w[tr_idx], w[val_idx]
    
    train_data = lgb.Dataset(X_tr, label=y_tr, weight=w_tr)
    valid_data = lgb.Dataset(X_val, label=y_val, weight=w_val, reference=train_data)
    
    model = lgb.train(
        params_lgb2,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=3000,
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING_ROUNDS),
            lgb.log_evaluation(100)
        ]
    )
    
    pred = model.predict(X_val)
    oof_lgb2[val_idx] = pred
    score = wmae_score(y_val, pred, w_val)
    scores_lgb2.append(score)
    models_lgb2.append(model)
    print(f"   WMAE*: {score:.4f}")

print(f"\n🎯 LightGBM-2 Mean WMAE*: {np.mean(scores_lgb2):.4f} ± {np.std(scores_lgb2):.4f}")
print(f"🎯 LightGBM-2 OOF WMAE*:  {wmae_score(y, oof_lgb2, w):.4f}")

In [ ]:
# ============================================================
# ПОИСК ОПТИМАЛЬНЫХ ВЕСОВ АНСАМБЛЯ (BLENDING)
# ============================================================
print("=" * 60)
print("ПОИСК ОПТИМАЛЬНЫХ ВЕСОВ BLENDING")
print("=" * 60)

best_wmae = float('inf')
best_weights = (0.5, 0.5)
results = []

for w1 in np.arange(0.0, 1.05, 0.05):
    w2 = 1 - w1
    blended_oof = w1 * oof_lgb1 + w2 * oof_lgb2
    wmae = wmae_score(y, blended_oof, w)
    results.append((w1, w2, wmae))
    if wmae < best_wmae:
        best_wmae = wmae
        best_weights = (w1, w2)

# Визуализация
plt.figure(figsize=(10, 4))
ws = [r[0] for r in results]
wmaes = [r[2] for r in results]
plt.plot(ws, wmaes, 'b-o', linewidth=2, markersize=6)
plt.axvline(best_weights[0], color='r', linestyle='--', label=f'Best: {best_weights[0]:.2f}')
plt.xlabel('Weight for LightGBM-1')
plt.ylabel('WMAE*')
plt.title('Blending Weight Optimization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\n✅ Лучшие веса: LGB1={best_weights[0]:.2f}, LGB2={best_weights[1]:.2f}")
print(f"✅ Лучший OOF WMAE*: {best_wmae:.4f}")

In [ ]:
# ============================================================
# ГЕНЕРАЦИЯ ФИНАЛЬНЫХ ПРЕДСКАЗАНИЙ
# ============================================================
print("=" * 60)
print("ГЕНЕРАЦИЯ ПРЕДСКАЗАНИЙ")
print("=" * 60)

# ВАЖНО: Усредняем предсказания ВСЕХ моделей (не только последней!)
test_pred_lgb1 = np.zeros(len(X_test))
for m in models_lgb1:
    test_pred_lgb1 += m.predict(X_test) / len(models_lgb1)

test_pred_lgb2 = np.zeros(len(X_test))
for m in models_lgb2:
    test_pred_lgb2 += m.predict(X_test) / len(models_lgb2)

# Финальный бленд с оптимальными весами
final_predictions = best_weights[0] * test_pred_lgb1 + best_weights[1] * test_pred_lgb2
final_predictions = np.clip(final_predictions, 0, None)  # доход не может быть отрицательным

print(f"📊 Статистика предсказаний:")
print(f"   Min: {final_predictions.min():.0f}")
print(f"   Max: {final_predictions.max():.0f}")
print(f"   Mean: {final_predictions.mean():.0f}")
print(f"   Std: {final_predictions.std():.0f}")

In [ ]:
# ============================================================
# ФОРМИРОВАНИЕ SUBMISSION
# ============================================================
submission = pd.DataFrame({
    'id': test_df['id'],
    'target': final_predictions
})

submission.to_csv('submission_improved.csv', index=False)
print(f"✅ Submission сохранён: {len(submission)} строк")
print(submission.head(10))

In [ ]:
# ============================================================
# ВАЖНОСТЬ ПРИЗНАКОВ
# ============================================================
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': models_lgb1[0].feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 10))
top_n = 25
plt.barh(importance['feature'].head(top_n)[::-1], 
         importance['importance'].head(top_n)[::-1],
         color='steelblue')
plt.xlabel('Importance (Gain)')
plt.title(f'Top {top_n} Feature Importance')
plt.tight_layout()
plt.show()

print("\n📋 Top 15 признаков:")
print(importance.head(15).to_string())

In [ ]:
# ============================================================
# ИТОГОВЫЕ РЕЗУЛЬТАТЫ
# ============================================================
print("=" * 60)
print("🏆 ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("=" * 60)
print(f"")
print(f"Baseline (mean) WMAE*:    {baseline_wmae:.2f}")
print(f"LightGBM-1 OOF WMAE*:     {wmae_score(y, oof_lgb1, w):.2f}")
print(f"LightGBM-2 OOF WMAE*:     {wmae_score(y, oof_lgb2, w):.2f}")
print(f"Ensemble OOF WMAE*:       {best_wmae:.2f}")
print(f"")
print(f"📈 Улучшение vs baseline: {((baseline_wmae - best_wmae) / baseline_wmae * 100):.1f}%")

# Сравнение с исходным решением (~40,150)
original_wmae = 40150  # примерный результат исходного решения
print(f"📈 Улучшение vs исходное: {((original_wmae - best_wmae) / original_wmae * 100):.2f}%")
print(f"   ({original_wmae:.0f} → {best_wmae:.0f})")

In [ ]:
# ============================================================
# SHAP (опционально - требует больше времени)
# ============================================================
# Раскомментируйте для SHAP анализа

# import shap
# 
# # Используем подвыборку для скорости
# sample_size = min(1000, len(X))
# X_sample = X.sample(sample_size, random_state=42)
# 
# explainer = shap.TreeExplainer(models_lgb1[0])
# shap_values = explainer.shap_values(X_sample)
# 
# plt.figure(figsize=(12, 8))
# shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
# plt.tight_layout()
# plt.show()

print("💡 SHAP анализ закомментирован для экономии времени")
print("   Раскомментируйте код выше для детального анализа")

In [ ]:
# ============================================================
# ФУНКЦИЯ РЕКОМЕНДАЦИЙ (для бизнес-применения)
# ============================================================
def generate_recommendations(predicted_income):
    """
    Генерация банковских рекомендаций на основе предсказанного дохода
    """
    if predicted_income < 30000:
        segment = "Базовый"
        products = ["Дебетовая карта с кэшбэком", "Микрокредит до 50,000 ₽"]
        credit_limit = min(predicted_income * 3, 50000)
    elif predicted_income < 80000:
        segment = "Стандарт"
        products = ["Кредитная карта до 300,000 ₽", "Потребительский кредит", "Накопительный счёт"]
        credit_limit = min(predicted_income * 6, 300000)
    elif predicted_income < 150000:
        segment = "Премиум"
        products = ["Премиальная карта", "Кредит до 1,000,000 ₽", "Инвестиционные продукты"]
        credit_limit = min(predicted_income * 8, 700000)
    else:
        segment = "Привилегия"
        products = ["Private Banking", "Ипотека", "ИИС", "Персональный менеджер"]
        credit_limit = min(predicted_income * 12, 2000000)
    
    return {
        "segment": segment,
        "predicted_income": predicted_income,
        "recommended_products": products,
        "suggested_credit_limit": credit_limit
    }

# Пример использования
sample_pred = final_predictions[0]
recommendation = generate_recommendations(sample_pred)
print(f"\n📋 Пример рекомендации:")
print(f"   Предсказанный доход: {recommendation['predicted_income']:.0f} ₽")
print(f"   Сегмент: {recommendation['segment']}")
print(f"   Рекомендуемые продукты: {', '.join(recommendation['recommended_products'])}")
print(f"   Рекомендуемый кредитный лимит: {recommendation['suggested_credit_limit']:.0f} ₽")

## 📝 Что ещё можно попробовать для улучшения:

1. **CatBoost / XGBoost** - добавить в ансамбль другие градиентные бустинги
2. **Optuna** - автоматический подбор гиперпараметров
3. **Target Encoding** - для категориальных признаков
4. **Stacking** - использовать мета-модель поверх базовых
5. **Pseudo-labeling** - использовать уверенные предсказания на тесте
6. **Больше FE** - взаимодействия, полиномиальные признаки